In [6]:
from dataclasses import dataclass, field
from enum import IntEnum
from typing import Self

import numpy as np
from scipy.optimize import linear_sum_assignment
# import pandas as pd

In [ ]:
class BDAEnum(IntEnum):
    """Base Enum for BDA Enums."""
    @property
    def text(self) -> str:
        """Returns the string representation of the field."""
        # ex. 0 -> "NO_DAMAGE" -> "no damage"
        return self.name.lower().replace("_", " ")
    
    @property
    def k_len(self):
        """Returns the total number of categories (K) in the Enum."""
        # NOTE: __members__ returns dictionary of Enum items
        return len(self.__class__.__members__)


class TargetType(BDAEnum):
    """IntEnum containing ordered collection of target types."""
    BRIDGES = 0
    MILITARY_EQUIPMENT = 7


class DamageBridge(BDAEnum):
    """IntEnum containing ordered collection of damage levels for bridges."""
    NO_DAMAGE = 0
    LIGHT_DAMAGE = 1
    MODERATE_DAMAGE = 2
    SEVERE_DAMAGE = 3
    DESTROYED = 4


class DamageMilitaryEquipment(BDAEnum):
    """IntEnum containing ordered collection of damage levels for military equipment."""
    NO_DAMAGE = 0
    DAMAGED = 1
    DESTROYED = 2


class Confidence(BDAEnum):
    """IntEnum containing ordered collection of confidence levels."""
    POSSIBLE = 0
    PROBABLE = 1
    CONFIRMED = 2


target_damage_map = {
    "bridges": {
        "target_type": TargetType.BRIDGES,
        "damage_category": DamageBridge
    },
    "military_equipment": {
        "target_type": TargetType.MILITARY_EQUIPMENT,
        "damage_category": DamageMilitaryEquipment
    }
}


@dataclass
class BoundingBox:
    """Represents a 2D bounding box."""
    xmin: int
    ymin: int
    xmax: int
    ymax: int
    
    # Declare "area" property, but don't require at initialization
    area: int = field(init=False)
    
    def __post_init__(self):
        """Performs additional setup after __init__ is complete."""
        #Calculate and store the area of the bounding box.
        self.area = (self.xmax - self.xmin) * (self.ymax - self.ymin)
    
    def calc_iou(self, box: Self) -> float:
        """Calculates the Intersection over Union (IoU) for two bounding boxes.
        
        Args:
            box: BoundingBox instance.
        
        Returns:
            Value of IoU
        """
        intersection = self.intersect_area(box)
        union = self.area + box.area - intersection
        
        if union == 0:
            return 0.0
        
        return intersection / union
    
    def intersect_area(self, box: Self) -> int:
        """Calculates intersecting area of two bounding boxes.
        
        Args:
            box: BoundingBox instance.
        
        Returns:
            The area of the overlap or zero if boxes do not overlap.
        """
        # Check if t1 and t2 are completely disjoint
        if box.xmax < self.xmin or \
            box.xmin > self.xmax or \
            box.ymin > self.ymax or \
            box.ymax < self.ymin:
            return 0

        # Determine intersecting rectangle:
        #     1. Find the min of ending x and y values
        #     2. Find the max of starting x and y values
        overlap_width = min(self.xmax, box.xmax) - max(self.xmin, box.xmin)
        overlap_height = min(self.ymax, box.ymax) - max(self.ymin, box.ymin)

        return overlap_width * overlap_height


@dataclass
class BDA:
    """Represents a Battle Damage Assessment report."""
    target_type: TargetType
    damage_category: BDAEnum
    confidence: Confidence
    logic: str
    box: BoundingBox
    ndarray: np.ndarray
    
    @classmethod
    def from_dict(cls, data: dict) -> Self:
        """Class factory method to create a BDA object from JSON."""
        logic = data["l"]
        box = BoundingBox(**data["bounding_box"])
        
        # Get inner dictionary from target_damage_map dictionary
        #     ex. td_map = { "target_type": ..., "damage_category": ... }
        td_map = target_damage_map[data["t"]]

        # Store BDA fields as IntEnum-subclassed objects
        target_type = td_map["target_type"]
        damage_category = td_map["damage_category"][data["d"].upper().replace(" ", "_")]
        confidence = Confidence[data["c"].upper()]
        
        # Convert BDA fields to numpy array for more efficient filtering/comparison
        ndarray = np.array([
            target_type,
            damage_category,
            confidence
        ])
        
        # Return instantiated class
        return cls(
            target_type=target_type,
            damage_category=damage_category,
            confidence=confidence,
            logic=logic,
            box=box,
            ndarray=ndarray
        )


class BDAReport:
    """Manages a collection of BDA objects."""
    def __init__(self, bdas: list[BDA]):
        self.bdas = bdas
        
        # Build a matrix of BDAs (N rows x 3 columns):
        #     Column 0: Target Type | Column 1: Damage | Column 2: Confidence
        self.matrix = np.array([bda.ndarray for bda in bdas])

    def filter_by_target(self, target_type: TargetType) -> list[BDA]:
        """Returns BDA objects matching a specific target type."""
        mask = self.matrix[:, 0] == target_type
        
        # TODO: Mask is now a numpy boolean filter. Keep track of that?
        
        # If mask is true for a given BDA, add BDA to resulting list
        return [bda for bda, m in zip(self.bdas, mask) if m]
        
    def filter_by_bda(self, reference_bda: BDA) -> list[BDA]:
        """Finds all BDAs that share the same target type as the reference."""
        return self.filter_by_target(reference_bda.target_type)

    def get_bda_matches(
        self, 
        R: Self, 
        min_iou: float = 0.001,
        w_d: float = 0.05, 
        w_c: float = 0.05
    ) -> list[tuple[BDA, BDA]]:
        """
        Pairs predictions to reference BDAs using the Hungarian Algorithm
        Primary metric is IoU. Normalized ordinal distance of damage and 
        confidence are used as weighted tiebreakers.
        
        Args:
            R: The BDAReport containing human assessments.
            min_iou: Minimum IoU required to consider a match valid.
            w_d: Weight of the damage penalty tiebreaker.
            w_c: Weight of the confidence penalty tiebreaker.
        
        Returns:
            List of matched BDAs.
        """
        matches = []
        max_cost = 1e5
        
        if len(R.bdas) == 0:
            return []
        
        # Find unique target types in R (ex. [bridge bridge tank bridge] -> [bridge tank])
        unique_target_types = np.unique(R.matrix[:, 0])
        
        # Iterate through each target type and apply Hungarian Algorithm
        for target_type in unique_target_types:
            t_enum = TargetType(target_type)
            
            # Look at objects with the same target type only
            P_filtered = self.filter_by_target(t_enum)
            R_filtered = R.filter_by_target(t_enum)
            
            n = len(P_filtered)
            m = len(R_filtered)
            
            if n == 0 or m == 0:
                continue
                
            # Initialize the N x M Multi-Objective Cost Matrix
            cost_matrix = np.zeros((n, m), dtype=float)
            
            # For every target in P, compare against R and build Cost Matrix
            for i, P_target in enumerate(P_filtered):
                for j, R_target in enumerate(R_filtered):
                    # Calculate the base cost (i.e. IoU between targets, inverted)
                    iou = P_target.box.calc_iou(R_target.box)
                    
                    if iou >= min_iou:
                        base_cost = 1.0 - iou
                        
                        # Subcost 1: Normalized Damage Cost (no longer subtracted from one)
                        #     K = number of damage levels for target_R
                        #         Dynamically find K by checking the length of the Enum
                        k = R_target.damage_category.k_len
                        c_d = abs(P_target.damage_category.value - R_target.damage_category.value) / (k - 1)
                        
                        # Subcost 2: Normalized Confidence Cost
                        # NOTE: Confidence always has 3 levels (0, 1, 2), so max distance is 2
                        c_c = abs(P_target.confidence.value - R_target.confidence.value) / 2.0
                            
                        # Total (weighted) cost
                        cost_matrix[i, j] = base_cost + (w_d * c_d) + (w_c * c_c)
                    else:
                        # IoU doesn't meet threshold IoU
                        cost_matrix[i, j] = max_cost

            # Run the Hungarian Algorithm on this specific subset (mathmagic)
            P_indices, R_indices = linear_sum_assignment(cost_matrix)

            # Assemble the matched pairs (filtering out invalid boxes)
            for p_idx, r_idx in zip(P_indices, R_indices):
                p_bda = P_filtered[p_idx]
                r_bda = R_filtered[r_idx]
                
                # Check the raw IoU to ensure they actually overlap
                # Necessary because linear_sum_assignment may still pair to a terrible match
                actual_iou = p_bda.box.calc_iou(r_bda.box)
                
                if actual_iou >= min_iou: 
                    matches.append((p_bda, r_bda))

        return matches



Let's test the match logic (thanks Gemini):

In [22]:
# ==========================================
# REFERENCE BDAs (Ground Truth - R)
# 10 Non-overlapping objects
# ==========================================
raw_R = [
    {"t": "bridges", "d": "destroyed", "c": "confirmed", "l": "GT1", "bounding_box": {"xmin": 10, "ymin": 10, "xmax": 50, "ymax": 50}},
    {"t": "bridges", "d": "severe_damage", "c": "probable", "l": "GT2", "bounding_box": {"xmin": 60, "ymin": 10, "xmax": 100, "ymax": 50}},
    {"t": "bridges", "d": "moderate_damage", "c": "possible", "l": "GT3", "bounding_box": {"xmin": 110, "ymin": 10, "xmax": 150, "ymax": 50}},
    {"t": "military_equipment", "d": "destroyed", "c": "confirmed", "l": "GT4", "bounding_box": {"xmin": 10, "ymin": 60, "xmax": 50, "ymax": 100}},
    {"t": "military_equipment", "d": "damaged", "c": "probable", "l": "GT5", "bounding_box": {"xmin": 60, "ymin": 60, "xmax": 100, "ymax": 100}},
    {"t": "military_equipment", "d": "no_damage", "c": "possible", "l": "GT6", "bounding_box": {"xmin": 110, "ymin": 60, "xmax": 150, "ymax": 100}},
    {"t": "military_equipment", "d": "destroyed", "c": "confirmed", "l": "GT7", "bounding_box": {"xmin": 10, "ymin": 110, "xmax": 50, "ymax": 150}},
    {"t": "military_equipment", "d": "damaged", "c": "probable", "l": "GT8", "bounding_box": {"xmin": 60, "ymin": 110, "xmax": 100, "ymax": 150}},
    {"t": "military_equipment", "d": "no_damage", "c": "possible", "l": "GT9", "bounding_box": {"xmin": 110, "ymin": 110, "xmax": 150, "ymax": 150}},
    {"t": "bridges", "d": "light_damage", "c": "confirmed", "l": "GT10", "bounding_box": {"xmin": 160, "ymin": 110, "xmax": 200, "ymax": 150}},
]

# ==========================================
# PREDICTED BDAs (VLM Output - P)
# 15 Objects Total
# ==========================================
raw_P = [
    # --- 8 True Positives (Minimally offset, perfectly matched assessments) ---
    {"t": "bridges", "d": "destroyed", "c": "confirmed", "l": "P1 matches GT1", "bounding_box": {"xmin": 12, "ymin": 12, "xmax": 48, "ymax": 48}},
    {"t": "bridges", "d": "severe_damage", "c": "probable", "l": "P2 matches GT2", "bounding_box": {"xmin": 58, "ymin": 12, "xmax": 102, "ymax": 48}},
    {"t": "bridges", "d": "moderate_damage", "c": "possible", "l": "P3 matches GT3", "bounding_box": {"xmin": 111, "ymin": 11, "xmax": 149, "ymax": 49}},
    {"t": "military_equipment", "d": "destroyed", "c": "confirmed", "l": "P4 matches GT4", "bounding_box": {"xmin": 10, "ymin": 62, "xmax": 50, "ymax": 102}},
    {"t": "military_equipment", "d": "damaged", "c": "probable", "l": "P5 matches GT5", "bounding_box": {"xmin": 62, "ymin": 62, "xmax": 98, "ymax": 98}},
    {"t": "military_equipment", "d": "no_damage", "c": "possible", "l": "P6 matches GT6", "bounding_box": {"xmin": 108, "ymin": 58, "xmax": 152, "ymax": 102}},
    {"t": "military_equipment", "d": "destroyed", "c": "confirmed", "l": "P7 matches GT7", "bounding_box": {"xmin": 10, "ymin": 110, "xmax": 50, "ymax": 155}},
    {"t": "military_equipment", "d": "damaged", "c": "probable", "l": "P8 matches GT8", "bounding_box": {"xmin": 60, "ymin": 115, "xmax": 100, "ymax": 150}},

    # --- 2 True Positives (Minimally offset, DIFFERENT assessments) ---
    # P9 matches GT9, but predicts 'Damaged/Confirmed' instead of 'No Damage/Possible'
    {"t": "military_equipment", "d": "damaged", "c": "confirmed", "l": "P9 matches GT9 (Diff Assess)", "bounding_box": {"xmin": 112, "ymin": 112, "xmax": 148, "ymax": 148}},
    # P10 matches GT10, but predicts 'Destroyed/Probable' instead of 'Light/Confirmed'
    {"t": "bridges", "d": "destroyed", "c": "probable", "l": "P10 matches GT10 (Diff Assess)", "bounding_box": {"xmin": 158, "ymin": 108, "xmax": 202, "ymax": 152}},

    # --- 5 Hallucinated Objects (False Positives) ---
    # Hallucination 1: EXACT same bounding box as GT1, but wrong target type (MilEq instead of Bridge).
    # This tests your `filter_by_target` method. It should NOT match GT1.
    {"t": "military_equipment", "d": "destroyed", "c": "confirmed", "l": "H1: Exact BB as GT1 (Wrong Target)", "bounding_box": {"xmin": 10, "ymin": 10, "xmax": 50, "ymax": 50}},
    
    # Hallucination 2: EXACT same bounding box as GT4, same target type, but terrible assessment.
    # This is a brilliant test: P4 is slightly offset but has perfect assessment. H2 has a perfect box but terrible assessment. 
    # Your tiebreaker weights (w_d, w_c) will decide who wins GT4!
    {"t": "military_equipment", "d": "no_damage", "c": "possible", "l": "H2: Exact BB as GT4 (Wrong Assess)", "bounding_box": {"xmin": 10, "ymin": 60, "xmax": 50, "ymax": 100}},

    # Hallucination 3: Encompasses both GT5 and GT6.
    # Tests IoU logic to see which GT it prioritizes based on overlap ratios.
    {"t": "military_equipment", "d": "damaged", "c": "probable", "l": "H3: Encompasses GT5 & GT6", "bounding_box": {"xmin": 50, "ymin": 50, "xmax": 160, "ymax": 110}},

    # Hallucinations 4 & 5: Wildly different bounding boxes.
    # Tests the `min_iou` thresholding. These should not match to anything.
    {"t": "bridges", "d": "destroyed", "c": "confirmed", "l": "H4: Wildly Different BB", "bounding_box": {"xmin": 800, "ymin": 800, "xmax": 900, "ymax": 900}},
    {"t": "military_equipment", "d": "destroyed", "c": "confirmed", "l": "H5: Wildly Different BB", "bounding_box": {"xmin": 900, "ymin": 100, "xmax": 950, "ymax": 150}}
]

R_list = [BDA.from_dict(r) for r in raw_R]
P_list = [BDA.from_dict(p) for p in raw_P]

R = BDAReport(R_list)
P = BDAReport(P_list)

for BDA1, BDA2 in P.get_bda_matches(R):
    print(f"{'=' * 160}\n{BDA1}\n{BDA2}")

BDA(target_type=<TargetType.BRIDGES: 0>, damage_category=<DamageBridge.DESTROYED: 4>, confidence=<Confidence.CONFIRMED: 2>, logic='P1 matches GT1', box=BoundingBox(xmin=12, ymin=12, xmax=48, ymax=48, area=1296), ndarray=array([0, 4, 2]))
BDA(target_type=<TargetType.BRIDGES: 0>, damage_category=<DamageBridge.DESTROYED: 4>, confidence=<Confidence.CONFIRMED: 2>, logic='GT1', box=BoundingBox(xmin=10, ymin=10, xmax=50, ymax=50, area=1600), ndarray=array([0, 4, 2]))
BDA(target_type=<TargetType.BRIDGES: 0>, damage_category=<DamageBridge.SEVERE_DAMAGE: 3>, confidence=<Confidence.PROBABLE: 1>, logic='P2 matches GT2', box=BoundingBox(xmin=58, ymin=12, xmax=102, ymax=48, area=1584), ndarray=array([0, 3, 1]))
BDA(target_type=<TargetType.BRIDGES: 0>, damage_category=<DamageBridge.SEVERE_DAMAGE: 3>, confidence=<Confidence.PROBABLE: 1>, logic='GT2', box=BoundingBox(xmin=60, ymin=10, xmax=100, ymax=50, area=1600), ndarray=array([0, 3, 1]))
BDA(target_type=<TargetType.BRIDGES: 0>, damage_category=<Dama

DELETEME: Every target has four components: t, d, c and l. Each component is in turn a string that is mapped to a corresponding ordinal (ex. "possible" == 0)

In [23]:
# Retrieve reference reports (containing string values)
# TODO: implement file retrieval and extraction functionality
r1 = {
    "t": "military_equipment",
    "d": "destroyed",
    "c": "confirmed",
    "l": "smokey",
    "bounding_box": {
        "xmin": 10,
        "ymin": 10,
        "xmax": 20,
        "ymax": 20
    }
}

r2 = {
    "t": "military_equipment",
    "d": "damaged",
    "c": "confirmed",
    "l": "decapitated",
    "bounding_box": {
        "xmin": 25,
        "ymin": 25,
        "xmax": 35,
        "ymax": 40
    }
}

R = [BDA.from_dict(r) for r in (r1, r2)]

for i, r in enumerate(R):
    print("=" * 80)
    print(f"REF TARGET {i}:\n")
    print(f"\t{r.target_type.text}: {r.target_type}")
    print(f"\t{r.damage_category.text}: {r.damage_category}")
    print(f"\t{r.confidence.text}: {r.confidence}")
    print(f"\tLogic: {r.logic}")
    print(f"\tNumpy Array: {r.ndarray}")
    print(f"\tBounding Box: {r.box}")
    print(f"\tBounding Box Area: {r.box.area}\n")


# =========================================================
# NOTE: Saved for later use, if necessary
# df = pd.DataFrame(R_str)

# # Create a function to apply row by row
# def encode_row(row):
#     return pd.Series({
#         't_encoded': map_damage[row['t']]['t_index'],
#         'd_encoded': map_damage[row['t']]['d'][row['d']],
#         'c_encoded': map_confidence[row['c']]
#     })

# # Apply the function and get your integer matrix
# R_df = df.apply(encode_row, axis=1)
# R_array = R_df.to_numpy()

# display(R_array)

REF TARGET 0:

	military equipment: 7
	destroyed: 2
	confirmed: 2
	Logic: smokey
	Numpy Array: [7 2 2]
	Bounding Box: BoundingBox(xmin=10, ymin=10, xmax=20, ymax=20, area=100)
	Bounding Box Area: 100

REF TARGET 1:

	military equipment: 7
	damaged: 1
	confirmed: 2
	Logic: decapitated
	Numpy Array: [7 1 2]
	Bounding Box: BoundingBox(xmin=25, ymin=25, xmax=35, ymax=40, area=150)
	Bounding Box Area: 150



DELETEME: Do the same thing to generate "VLM-produced" P.

In [25]:
p1 = {
    "t": "bridges",
    "d": "severe damage",
    "c": "probable",
    "l": "...",
    "bounding_box": {
        "xmin": 0,
        "ymin": 0,
        "xmax": 0,
        "ymax": 0
    }
}

p2 = {
    "t": "military_equipment",
    "d": "destroyed",
    "c": "confirmed",
    "l": "...",
    "bounding_box": {
        "xmin": 12,
        "ymin": 12,
        "xmax": 20,
        "ymax": 20
    }
}

p3 = {
    "t": "bridges",
    "d": "no damage",
    "c": "probable",
    "l": "...",
    "bounding_box": {
        "xmin": 0,
        "ymin": 0,
        "xmax": 0,
        "ymax": 0
    }
}

p4 = {
    "t": "military_equipment",
    "d": "damaged",
    "c": "confirmed",
    "l": "...",
    "bounding_box": {
        "xmin": 27,
        "ymin": 27,
        "xmax": 36,
        "ymax": 40
    }
}

P = [BDA.from_dict(p) for p in (p1, p2, p3, p4)]

for i, p in enumerate(P):
    print("=" * 80)
    print(f"PRED TARGET {i}:\n")
    print(f"\t{p.target_type.text}: {p.target_type}")
    print(f"\t{p.damage_category.text}: {p.damage_category}")
    print(f"\t{p.confidence.text}: {p.confidence}")
    print(f"\tLogic: {p.logic}")
    print(f"\tNumpy Array: {p.ndarray}")
    print(f"\tBounding Box: {p.box}")
    print(f"\tBounding Box Area: {p.box.area}\n")



PRED TARGET 0:

	bridges: 0
	severe damage: 3
	probable: 1
	Logic: ...
	Numpy Array: [0 3 1]
	Bounding Box: BoundingBox(xmin=0, ymin=0, xmax=0, ymax=0, area=0)
	Bounding Box Area: 0

PRED TARGET 1:

	military equipment: 7
	destroyed: 2
	confirmed: 2
	Logic: ...
	Numpy Array: [7 2 2]
	Bounding Box: BoundingBox(xmin=12, ymin=12, xmax=20, ymax=20, area=64)
	Bounding Box Area: 64

PRED TARGET 2:

	bridges: 0
	no damage: 0
	probable: 1
	Logic: ...
	Numpy Array: [0 0 1]
	Bounding Box: BoundingBox(xmin=0, ymin=0, xmax=0, ymax=0, area=0)
	Bounding Box Area: 0

PRED TARGET 3:

	military equipment: 7
	damaged: 1
	confirmed: 2
	Logic: ...
	Numpy Array: [7 1 2]
	Bounding Box: BoundingBox(xmin=27, ymin=27, xmax=36, ymax=40, area=117)
	Bounding Box Area: 117



DELETEME: Find the best matches. As a test, only doing the first reference object, [7, 2, 2].

In [ ]:
candidates = P[R[0, 0] == P[:, 0]]

print("Candidates:")
display(candidates)

# Yeah, hardcoded for now
K = len(map_damage["military_equipment"]["d"].keys())

s_d = 1 - (abs(candidates[:, 1] - R[0, 1]) / (K - 1))
s_c = 1 - (abs(candidates[:, 2] - R[0, 2]) / 2)

w_d, w_c = (1, 1)
s = w_d * s_d + w_c * s_c

print("Match Values:")
display(s)

print("Best Match:")
display(candidates[s.argmax()])

Candidates:


array([[7, 2, 2],
       [7, 0, 1],
       [7, 1, 2]])

Match Values:


array([2. , 0.5, 1.5])

Best Match:


array([7, 2, 2])